In [1]:
# Late Fusion: RGB + Bone Stream Violence Detection (RWF-2000)
#
# Identical to 06_fusion_RGB_jointStream.ipynb but uses the
# BONE stream checkpoint and GenSkeFeat feats=['b'] instead of ['j'].
#
# Models
# | Model        | Architecture | Individual Accuracy |
# |--------------|-------------|---------------------|
# | RGB Stream   | R3D-18      | 88.00%              |
# | Bone Stream  | 2s-AGCN     | 83.50%              |

# ── 1. Mount & install ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install /content/drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl -q
!pip install mmengine -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 25.2 MB/s eta 0:00:00


In [10]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pickle
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [11]:
# ── 3. Load RGB predictions (already saved from previous run) ─────────────────
RGB_PROBS_PATH  = '/content/drive/MyDrive/violence-detection/checkpoints/rgb_v2/rgb_v2_val_probs.npy'
RGB_LABELS_PATH = '/content/drive/MyDrive/violence-detection/checkpoints/rgb_v2/rgb_v2_val_labels.npy'

rgb_probs_arr  = np.load(RGB_PROBS_PATH)
rgb_labels_arr = np.load(RGB_LABELS_PATH)

print("RGB probs shape:  ", rgb_probs_arr.shape)
print("RGB only accuracy:", accuracy_score(rgb_labels_arr,
                                           np.argmax(rgb_probs_arr, axis=1)))

RGB probs shape:   (400, 2)
RGB only accuracy: 0.88


In [12]:
# ── 4. Load Bone predictions (already saved from 3-stream notebook) ───────────
BONE_PATH = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-bone/bone_val_predictions.pkl'

with open(BONE_PATH, 'rb') as f:
    bone_preds = pickle.load(f)

bone_probs_arr  = np.array([p['pred_score'].numpy() for p in bone_preds])
bone_labels_arr = np.array([p['gt_label'].item()    for p in bone_preds])

print("Bone probs shape: ", bone_probs_arr.shape)
print("Bone only accuracy:", accuracy_score(bone_labels_arr,
                                            np.argmax(bone_probs_arr, axis=1)))

Bone probs shape:  (400, 2)
Bone only accuracy: 0.835


In [13]:
# ── 5. Check alignment ────────────────────────────────────────────────────────
print("Labels match:", np.all(rgb_labels_arr == bone_labels_arr))
assert np.all(rgb_labels_arr == bone_labels_arr), "Label mismatch!"
labels = bone_labels_arr
print("Streams aligned ✅")

Labels match: True
Streams aligned ✅


In [14]:
# ── 6. RGB + Bone grid search fusion ─────────────────────────────────────────
print("=" * 55)
print(f"{'Weights (RGB / Bone)':<30} {'Acc':>10} {'F1':>10}")
print("=" * 55)

best_acc_rb   = -1
best_f1_rb    = -1
best_w_rgb    = -1
best_preds_rb = None

for w_rgb in np.arange(0.0, 1.05, 0.05):
    w_rgb  = round(float(w_rgb), 2)
    w_bone = round(1.0 - w_rgb, 2)
    fused  = w_rgb * rgb_probs_arr + w_bone * bone_probs_arr
    preds  = np.argmax(fused, axis=1)
    acc    = accuracy_score(labels, preds)
    f1     = f1_score(labels, preds)
    print(f"RGB {w_rgb:.2f} + Bone {w_bone:.2f}          {acc:.4f}     {f1:.4f}")
    if acc > best_acc_rb or (acc == best_acc_rb and f1 > best_f1_rb):
        best_acc_rb   = acc
        best_f1_rb    = f1
        best_w_rgb    = w_rgb
        best_preds_rb = preds

print("=" * 55)
print(f"Best: RGB {best_w_rgb:.2f} + Bone {round(1-best_w_rgb, 2):.2f}")
print(f"Best accuracy : {best_acc_rb:.4f}  →  {best_acc_rb*100:.2f}%")
print(f"Best F1       : {best_f1_rb:.4f}")

Weights (RGB / Bone)                  Acc         F1
RGB 0.00 + Bone 1.00          0.8350     0.8272
RGB 0.05 + Bone 0.95          0.8350     0.8272
RGB 0.10 + Bone 0.90          0.8400     0.8325
RGB 0.15 + Bone 0.85          0.8475     0.8407
RGB 0.20 + Bone 0.80          0.8525     0.8460
RGB 0.25 + Bone 0.75          0.8500     0.8438
RGB 0.30 + Bone 0.70          0.8525     0.8468
RGB 0.35 + Bone 0.65          0.8550     0.8497
RGB 0.40 + Bone 0.60          0.8600     0.8542
RGB 0.45 + Bone 0.55          0.8650     0.8608
RGB 0.50 + Bone 0.50          0.8850     0.8827
RGB 0.55 + Bone 0.45          0.8825     0.8816
RGB 0.60 + Bone 0.40          0.8825     0.8822
RGB 0.65 + Bone 0.35          0.8800     0.8800
RGB 0.70 + Bone 0.30          0.8800     0.8806
RGB 0.75 + Bone 0.25          0.8850     0.8861
RGB 0.80 + Bone 0.20          0.8825     0.8840
RGB 0.85 + Bone 0.15          0.8850     0.8861
RGB 0.90 + Bone 0.10          0.8825     0.8840
RGB 0.95 + Bone 0.05          0.880

In [15]:
# ── 7. Detailed report for best fusion ───────────────────────────────────────
print(f"\n=== RGB {best_w_rgb:.2f} + Bone {round(1-best_w_rgb,2):.2f} (Best) ===")
print(classification_report(labels, best_preds_rb,
                             target_names=['NonFight', 'Fight']))


=== RGB 0.75 + Bone 0.25 (Best) ===
              precision    recall  f1-score   support

    NonFight       0.89      0.88      0.88       200
       Fight       0.88      0.90      0.89       200

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.88       400
weighted avg       0.89      0.89      0.88       400

